# Practical 05 — Hyperspectral Satellite Imaging & Spectroscopy
**PLNT 7620 / Part 4 – Satellite Remote Sensing · University of Manitoba · November 2026**  
**Digital AgroEcosystems Lab · Instructor: Dr. Nasem Badreldin**

**Practical prepared by Dr. Zeinab Dastgheib**, University of Manitoba.

This practical uses **measured laboratory soil reflectance spectra** (Pascal dataset) to learn the spectroscopy workflow that underpins hyperspectral remote sensing: raw spectra → exploratory analysis → quality control → spectral preprocessing → machine-learning prediction. An **optional NASA EMIT satellite activity** at the end illustrates the difference between laboratory and spaceborne reflectance measurements.

**The shared teaching datasets are read-only for this practical.** Save your notebook as your own Drive copy, and save outputs in your own Drive folder. Do not upload input datasets, credentials, or unapproved sample locations to this public GitHub repository.

## Start here — access the instructor's shared Google Drive data

**Instructor-provided folder:** [Practical 05 spectroscopy files](https://drive.google.com/drive/folders/1MZeno2uHjEn-hFJTYknID5yaUFFbFjPE?usp=sharing).

1. Open the folder while signed into the Google account you use for Colab. Confirm you have permission to view and download the files.
2. Your instructor's files are located at **My Drive → PLNT 7620 → Data → SoilSpec**. The default path is `/content/drive/MyDrive/PLNT 7620/Data/SoilSpec`. If it is under **Shared with me**, the folder may not appear in Colab until added to My Drive. The setup cell first checks your mounted Drive and then tries downloading the instructor's shared teaching files directly into Colab. A Drive shortcut is optional; some shortcuts are not exposed as normal directories in Colab. If downloading is restricted, place the three Pascal Excel files in a folder in your own My Drive and set `INPUT_DIR_OVERRIDE`.
3. Open this notebook from the GitHub **Open in Colab** link, select **File → Save a copy in Drive**, and run the configuration cell first. Authorize the Drive mount when prompted.
4. Your figures and tables will be saved under **`MyDrive/PLNT7620_Practical_05_Outputs`** (never in the instructor's source folder).
5. Files used by the original lesson: `Pascal_Final.xlsx` (sheet `Sheet1`), `Pascal_Fixed_Sample_and_Fold_Assignments.xlsx` (sheet `Assignments`), and optional `Pascal_Color_Coef_No land name.xlsx` for Part 6B. The modeling sections require the assignment workbook to match the input sample IDs exactly. **If a file is not included in the shared folder, ask the instructor; do not create or fabricate missing research measurements or model splits.**

**Learning outcomes:** interpret VNIR–SWIR signatures (360–2500 nm), identify noisy wavelengths and potential outliers, apply SG/SNV/MSC methods, evaluate support vector regression (SVR), and understand limitations of transferring laboratory signatures to satellite observations.

### 0. Configure Drive access and file paths — run first

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd

SHARED_FOLDER_URL = 'https://drive.google.com/drive/folders/1MZeno2uHjEn-hFJTYknID5yaUFFbFjPE?usp=sharing'
SHARED_FOLDER_NAME = 'PLNT7620_05_Shared_Data'  # optional shortcut name in My Drive
INPUT_DIR_OVERRIDE = ''  # optional override; default finds MyDrive/PLNT 7620/Data/SoilSpec
OUTPUT_FOLDER_NAME = 'PLNT7620_Practical_05_Outputs'

try:
    from google.colab import drive
    drive.mount('/content/drive')
    DRIVE_ROOT = Path('/content/drive/MyDrive')
except ImportError:
    DRIVE_ROOT = Path.cwd()

# Outputs belong to the student, never the source folder.
STUDENT_OUTPUT_DIR = DRIVE_ROOT / OUTPUT_FOLDER_NAME
STUDENT_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print('Student output directory:', STUDENT_OUTPUT_DIR)

def locate_pascal_folder(folder):
    """Return actual folder with Pascal_Final.xlsx; do not substitute colour tables."""
    if not folder.is_dir():
        return None
    for location in [folder] + [folder / x for x in ('Data', 'data', 'Inputs', 'inputs', 'SoilSpec')]:
        if location.is_dir() and any(location.glob('Pascal_Final*.xlsx')):
            return location
    return None

if INPUT_DIR_OVERRIDE.strip():
    candidates = [Path(INPUT_DIR_OVERRIDE).expanduser()]
else:
    candidates = [
        DRIVE_ROOT / 'PLNT 7620' / 'Data' / 'SoilSpec',  # verified Drive folder hierarchy
        DRIVE_ROOT / 'Data' / 'SoilSpec',
        DRIVE_ROOT / 'SoilSpec',
        DRIVE_ROOT / SHARED_FOLDER_NAME,
        DRIVE_ROOT / 'Soil_Spectroscopy_Teaching',
        DRIVE_ROOT / '05. Hyperspectral Satellite Imaging & Spectroscopy',
        DRIVE_ROOT,
    ]
    # Other locations including shared drives and a one-level different root.
    for root in (DRIVE_ROOT, Path('/content/drive/Shareddrives')):
        if root.is_dir():
            for child in root.iterdir():
                if child.is_dir():
                    candidates.extend((child / 'Data' / 'SoilSpec', child / 'SoilSpec', child))

INPUT_DIR = next((found for candidate in candidates
                  if (found := locate_pascal_folder(candidate)) is not None), None)

if INPUT_DIR is None and not INPUT_DIR_OVERRIDE.strip():
    # Download from the instructor's existing shared link to temporary Colab storage.
    # This requires that the account/link permits access; nothing is posted to GitHub.
    import subprocess, sys
    try:
        import gdown
    except ImportError:
        subprocess.check_call([sys.executable, '-m', 'pip', '-q', 'install', 'gdown'])
        import gdown
    download_dir = Path('/content/PLNT7620_Practical_05_Inputs')
    download_dir.mkdir(parents=True, exist_ok=True)
    print('Pascal workbook not found on mounted Drive. Attempting shared-folder download...')
    try:
        downloaded = gdown.download_folder(url=SHARED_FOLDER_URL,
                                            output=str(download_dir),
                                            quiet=False, remaining_ok=True)
        print('Files downloaded:', len(downloaded or []))
    except Exception as exc:
        print('Shared-folder download did not succeed:', type(exc).__name__, str(exc)[:400])
    INPUT_DIR = locate_pascal_folder(download_dir)
    if INPUT_DIR is None and download_dir.is_dir():
        for subfolder in download_dir.iterdir():
            if subfolder.is_dir():
                INPUT_DIR = locate_pascal_folder(subfolder)
                if INPUT_DIR is not None:
                    break

if INPUT_DIR is None:
    raise FileNotFoundError(
        'Pascal_Final.xlsx is not accessible in Colab. The screenshot shows Data / SoilSpec, '
        'but the breadcrumb does not establish an accessible /content/drive path. '
        'If the folder is restricted, download the three Pascal Excel workbooks from '
        'the shared link, upload them to a folder in your own My Drive, and set '
        'INPUT_DIR_OVERRIDE to its full /content/drive/MyDrive/... path. '
        'Original folder: ' + SHARED_FOLDER_URL
    )

def find_input_file(filename, pattern=None):
    """Find original input workbook in selected folder or standard input subfolder."""
    locations = [INPUT_DIR] + [INPUT_DIR / x for x in ('Data', 'data', 'Inputs', 'inputs')]
    matches = [p / filename for p in locations if (p / filename).is_file()]
    if matches:
        return matches[0]
    if pattern:
        alternatives = [f for p in locations if p.is_dir() for f in p.glob(pattern)]
        if len(alternatives) == 1:
            return alternatives[0]
        if len(alternatives) > 1:
            raise FileNotFoundError(f'Ambiguous {filename}: {alternatives}')
    raise FileNotFoundError(f'Missing {filename} in {INPUT_DIR}. Check shared-folder permissions and contents.')

def normalize_spectral_columns(frame):
    """Accept Excel headers 360 and X360 without altering measured values."""
    frame = frame.copy()
    frame.columns = frame.columns.map(str).str.strip()
    renamed = {col: col[1:] for col in frame if col.startswith('X') and col[1:].isdigit()
               and col[1:] not in frame.columns}
    return frame.rename(columns=renamed)

print('Instructor input folder:', INPUT_DIR)
print('Available Excel files:', [p.name for p in INPUT_DIR.glob('*.xlsx')])
print('Student exports will be saved to:', STUDENT_OUTPUT_DIR)


In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap, Normalize
from matplotlib.cm import ScalarMappable
from IPython.display import display

# Shared data are configured in the setup cell above; student outputs are separate.
project_dir = INPUT_DIR
file_path = find_input_file('Pascal_Final.xlsx', pattern='Pascal_Final*.xlsx')
output_dir = STUDENT_OUTPUT_DIR / 'Part1_Outputs'
output_dir.mkdir(parents=True, exist_ok=True)
print('Input file:', file_path)
print('Output folder:', output_dir)

### Step 2 — Read the spectral data

Each **row** is one sample; each wavelength column is a measured reflectance value; `TOM_%LOI` is the laboratory-measured response. Excel may store wavelength headers as integers, so convert **column names only** to strings.

In [ ]:
data = pd.read_excel(file_path, sheet_name='Sheet1')
data = normalize_spectral_columns(data)
data = data.rename(columns={'TOM_%LOI': 'TOM'})

print('Samples read:', len(data))
print('Total columns:', data.shape[1])
print('First sample IDs:', data['id'].head().tolist() if 'id' in data.columns else '(IDs absent)')
print('Example column names:', data.columns[:15].tolist())
display(data.iloc[:10, :15])

### Step 3 — Choose wavelengths and prepare complete raw samples

The original R code uses wavelengths **360–2500 nm (1 nm steps)**. The original script keeps complete rows only; this is **not outlier removal**.

In [ ]:
wavelengths = np.arange(360, 2501)  # Change to np.arange(360, 701) for VIS only.
wavelength_cols = [str(w) for w in wavelengths]

missing = [name for name in wavelength_cols + ['TOM'] if name not in data.columns]
if missing:
    raise ValueError(f'Missing required columns: {missing[:12]}')

# Preserve IDs and original Excel row order; use row numbers only if IDs are absent.
work = data[['id'] + wavelength_cols + ['TOM']].copy() if 'id' in data.columns else data[wavelength_cols + ['TOM']].copy()
if 'id' not in work.columns:
    work.insert(0, 'id', np.arange(1, len(work) + 1))

# Convert measured values to numeric; invalid values become missing.
work[wavelength_cols + ['TOM']] = work[wavelength_cols + ['TOM']].apply(
    pd.to_numeric, errors='coerce'
)
work = work.replace([np.inf, -np.inf], np.nan)
rows_before = len(work)
work = work.dropna(subset=['id'] + wavelength_cols + ['TOM']).copy()
if work.empty:
    raise ValueError('No complete samples available for plotting.')

spectral_matrix = work[wavelength_cols].to_numpy(dtype=float)
tom_values = work['TOM'].to_numpy(dtype=float)
sample_ids = work['id'].astype(str).to_numpy()

print('Samples read:', rows_before)
print('Complete samples plotted:', len(work))
print('Rows excluded due to missing/nonfinite values:', rows_before - len(work))
print('Number of wavelengths:', len(wavelengths))
print('TOM range: {:.2f}–{:.2f} % LOI'.format(tom_values.min(), tom_values.max()))

### Step 4 — Figure 1: Raw spectra in blue

Plot each complete sample in its original Excel order, all in **blue**. The curves have not been smoothed, transformed, or colored by TOM.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 7))
for spectrum in spectral_matrix:  # Original order: no sorting.
    ax.plot(wavelengths, spectrum, color='blue', linewidth=0.45, alpha=0.9)

ax.set_title('Raw Soil Reflectance Spectra - Original Sample Order', fontweight='bold', fontsize=14)
ax.text(0.5, 1.01, 'All spectra shown in blue (original Excel row order)',
        transform=ax.transAxes, ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Reflectance')
ax.grid(color='#dddddd', alpha=0.7)
fig.tight_layout()
output_raw = output_dir / f'Figure1_Raw_Spectra_Original_Order_{wavelengths.min()}-{wavelengths.max()}.png'
fig.savefig(output_raw, dpi=300)
plt.show()
print('Saved:', output_raw)

Each curve represents one soil sample. Compare changes in reflectance with wavelength and differences among samples.

### Step 5 — Figure 2: Raw spectra colored by TOM

Sort **a copy** of the samples by increasing TOM, so higher-TOM curves are drawn later and remain visible when lines overlap. Use the same eight-color gradient as the original R script: blue → cyan → green → yellow → orange → red.

In [ ]:
# Stable sorting preserves original relative order when TOM values tie.
order_tom = np.argsort(tom_values, kind='stable')
spectral_matrix_sorted = spectral_matrix[order_tom, :]
tom_values_sorted = tom_values[order_tom]
sample_ids_sorted = sample_ids[order_tom]

r_colors = ['#211C9B', '#1259FF', '#00C5FF', '#00D98B',
            '#E6EF20', '#FF9C1A', '#F42336', '#A50026']
cmap = LinearSegmentedColormap.from_list('Pascal_TOM', r_colors, N=256)
normalizer = Normalize(vmin=float(tom_values.min()), vmax=float(tom_values.max()))

fig, ax = plt.subplots(figsize=(11, 7))
for spectrum, tom in zip(spectral_matrix_sorted, tom_values_sorted):
    ax.plot(wavelengths, spectrum, color=cmap(normalizer(tom)),
            linewidth=0.45, alpha=0.9)

ax.set_title('Raw Soil Reflectance Spectra Colored by TOM', fontweight='bold', fontsize=14)
ax.text(0.5, 1.01, 'Samples drawn in ascending TOM order',
        transform=ax.transAxes, ha='center', va='bottom', fontsize=10)
ax.set_xlabel('Wavelength (nm)')
ax.set_ylabel('Reflectance')
ax.grid(color='#dddddd', alpha=0.7)
cbar = fig.colorbar(ScalarMappable(norm=normalizer, cmap=cmap), ax=ax, pad=0.02)
cbar.set_label('TOM (% LOI)')
fig.tight_layout()
output_tom = output_dir / f'Figure2_Spectra_Colored_by_TOM_{wavelengths.min()}-{wavelengths.max()}.png'
fig.savefig(output_tom, dpi=300)
plt.show()
print('Saved:', output_tom)

# The sorted copy is used only for drawing: original data remain unchanged.
assert len(spectral_matrix_sorted) == len(spectral_matrix)
assert np.array_equal(np.sort(tom_values_sorted), np.sort(tom_values))

The color bar shows TOM measured in **% LOI**. Inspect which spectral regions separate low-TOM (blue) from high-TOM (red) samples. The colors represent observed TOM—not predictions—and a visible relationship alone does not prove a reliable predictive model.

# Part 2: TOM Exploratory Analysis

TOM transformations, sample SD, adjusted skewness, summary statistics, histograms, boxplots, and violin plots. Run cells from top to bottom.

In [ ]:
"""Part 2: Exploratory analysis of TOM (% LOI).
Run after Part 1, or standalone.
"""
from pathlib import Path
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# 1. FILE PATHS — read instructor data; save results in your Drive.
project_dir = INPUT_DIR
file_path = find_input_file('Pascal_Final.xlsx', pattern='Pascal_Final*.xlsx')
output_dir = STUDENT_OUTPUT_DIR / 'Part2_Outputs'
output_dir.mkdir(parents=True, exist_ok=True)


## Read and inspect TOM

Inspect the TOM column, count missing or invalid entries, and keep only finite TOM values for this exploratory analysis.

In [ ]:
# 2. READ AND VALIDATE TOM -----------------------------------------------------
data = pd.read_excel(file_path, sheet_name='Sheet1')
data = normalize_spectral_columns(data)
if 'TOM_%LOI' in data.columns:
    data = data.rename(columns={'TOM_%LOI': 'TOM'})
if 'TOM' not in data.columns:
    raise ValueError('TOM or TOM_%LOI column not found in Excel.')
tom_raw = pd.to_numeric(data['TOM'], errors='coerce').to_numpy(dtype=float)
missing = int(np.isnan(tom_raw).sum())
nonfinite = int((np.isinf(tom_raw)).sum())
tom = tom_raw[np.isfinite(tom_raw)]
print('Input file:', file_path.name)
print('Total rows:', len(tom_raw))
print('Missing or non-numeric TOM values:', missing)
print('Non-finite TOM values (Inf/-Inf):', nonfinite)
print('Samples included in TOM analysis:', len(tom))
print('Negative TOM values:', int((tom < 0).sum()))
print('Zero TOM values:', int((tom == 0).sum()))
if len(tom) < 3:
    raise ValueError('At least three finite TOM values are needed.')
if np.any(tom < 0):
    warnings.warn('Negative TOM detected. SQRT and log(TOM + 1) results exclude invalid values.')

## Calculate TOM transformations

Prepare raw TOM, square-root TOM, and log(TOM + 1). The transformed versions require TOM ≥ 0.

In [ ]:
# 3. RESPONSE TRANSFORMATIONS -------------------------------------------------
raw_values = tom.copy()
nonnegative_tom = tom[tom >= 0]
sqrt_values = np.sqrt(nonnegative_tom)
log_values = np.log1p(nonnegative_tom)   # exactly log(TOM + 1), NOT log(TOM)

## Compute summary statistics and export Excel

Calculate min, max, mean, median, sample standard deviation, CV%, and skewness on each scale. Values printed are rounded; Excel retains full precision.

In [ ]:
# 4. SUMMARY STATISTICS -------------------------------------------------------
def adjusted_skewness(x):
    """Match the exact adjusted Fisher-Pearson formula in the supplied R script."""
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if n < 3:
        return np.nan
    sd = np.std(x, ddof=1)  # matches R sd()
    if sd == 0:
        return np.nan
    z = (x - np.mean(x)) / sd
    return (n / ((n - 1) * (n - 2))) * np.sum(z**3)


def summarize_tom(x, label, total_valid, calculate_cv=True):
    x = np.asarray(x, dtype=float)
    x = x[np.isfinite(x)]
    n = len(x)
    if not n:
        return dict(Transformation=label, N=0, Excluded_for_Transform=total_valid,
                    Min=np.nan, Max=np.nan, Mean=np.nan, Median=np.nan,
                    SD=np.nan, CV_percent=np.nan, Skewness=np.nan)
    mu = float(np.mean(x))
    sd = float(np.std(x, ddof=1)) if n > 1 else np.nan
    cv = (100 * sd / mu) if calculate_cv and mu != 0 else np.nan
    return dict(Transformation=label, N=n, Excluded_for_Transform=total_valid-n,
                Min=float(np.min(x)), Max=float(np.max(x)), Mean=mu,
                Median=float(np.median(x)), SD=sd, CV_percent=cv,
                Skewness=adjusted_skewness(x))

series = [('Raw TOM (%)', raw_values),
          ('sqrt(TOM)', sqrt_values),
          ('log(TOM + 1)', log_values)]
summary_table = pd.DataFrame([summarize_tom(x, label, len(tom)) for label, x in series])
print('\nSummary statistics (display rounded; Excel retains full precision):')
print(summary_table.round({'Min':3,'Max':3,'Mean':3,'Median':3,'SD':3,'CV_percent':2,'Skewness':3}).to_string(index=False))
summary_file = output_dir / 'Part2_TOM_Summary_Statistics.xlsx'
with pd.ExcelWriter(summary_file) as writer:
    summary_table.to_excel(writer, sheet_name='TOM_Statistics', index=False)
print('Saved:', summary_file)

## Figure 1: Histograms

Histograms show how the distribution of TOM changes under square-root and log transformations. Compare shape within each panel, not absolute axis values across panels.

In [ ]:
# 5. FIGURE 1 — HISTOGRAMS ----------------------------------------------------
plt.rcParams.update({'font.size': 11})
fig, axes = plt.subplots(1, 3, figsize=(13, 5), constrained_layout=True)
for ax, (name, values) in zip(axes, series):
    ax.hist(values, bins=18, color='steelblue', edgecolor='white')
    ax.set_title(name)
    ax.set_xlabel('TOM (panel-specific scale)')
    ax.set_ylabel('Number of samples')
    ax.grid(False)
fig.suptitle('TOM Distribution: Raw vs Square Root vs log(TOM + 1)', fontsize=13, fontweight='bold')
hist_file = output_dir / 'Part2_TOM_Histograms.png'
fig.savefig(hist_file, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', hist_file)

## Figure 2: Boxplots

Boxplots show median, quartiles, and potential extreme values. Red points are potential outliers for investigation, not automatically removed observations.

In [ ]:
# 6. FIGURE 2 — BOXPLOTS ------------------------------------------------------
fig, axes = plt.subplots(1, 3, figsize=(13, 5), constrained_layout=True)
for ax, (name, values) in zip(axes, series):
    if len(values):
        ax.boxplot(values, widths=.4, patch_artist=True,
                   boxprops={'facecolor':'steelblue','alpha':.65},
                   flierprops={'marker':'o','markerfacecolor':'red','markeredgecolor':'red','markersize':4})
    ax.set_title(name)
    ax.set_ylabel('TOM (panel-specific scale)')
    ax.set_xticks([])
    ax.grid(False)
fig.suptitle('TOM Boxplots: Raw vs Square Root vs log(TOM + 1)', fontsize=13, fontweight='bold')
fig.text(.5, 1.05, 'Red points indicate boxplot outlier candidates, NOT removed samples', ha='center', fontsize=10)
box_file = output_dir / 'Part2_TOM_Boxplots.png'
fig.savefig(box_file, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', box_file)

## Figure 3: Violin plots with sample points

Violin plots visualize estimated density, with every measured TOM value displayed as a dot. The Python density may differ slightly from ggplot2 because KDE bandwidth rules differ.

In [ ]:
# 7. FIGURE 3 — VIOLIN WITH INDIVIDUAL POINTS --------------------------------
# Gaussian KDE is analogous to ggplot2 geom_violin; bandwidths may differ.
import numpy as np
import matplotlib.pyplot as plt
from scipy.stats import gaussian_kde

# Ensure all required variables are locally reconstructed if previous cells weren't run
if 'tom' not in globals():
    raise NameError("The variable 'tom' is not defined. Please run the cells in 'Step 2' and 'Step 3' first to load the TOM data.")

raw_values = tom.copy()
nonnegative_tom = tom[tom >= 0]
sqrt_values = np.sqrt(nonnegative_tom)
log_values = np.log1p(nonnegative_tom)

series = [('Raw TOM (%)', raw_values),
          ('sqrt(TOM)', sqrt_values),
          ('log(TOM + 1)', log_values)]

rng = np.random.default_rng(123)  # repeatable jitter; does NOT modify sample ordering
fig, axes = plt.subplots(1, 3, figsize=(13, 5), constrained_layout=True)
for ax, (name, values) in zip(axes, series):
    if len(values) > 1 and np.std(values) > 0:
        y_grid = np.linspace(min(values) - .12*np.ptp(values), max(values) + .12*np.ptp(values), 300)
        density = gaussian_kde(values)(y_grid)
        density = density / density.max() * .38
        ax.fill_betweenx(y_grid, -density, density, facecolor='skyblue', edgecolor='steelblue', alpha=.5)
    x_jitter = rng.uniform(-.09, .09, size=len(values))
    ax.scatter(x_jitter, values, s=10, alpha=.65, color='navy')
    ax.set_xlim(-.55, .55)
    ax.set_ylim(y_grid.min() - 0.05*np.ptp(y_grid),
                y_grid.max() + 0.05*np.ptp(y_grid))
    ax.set_xticks([])
    ax.set_ylabel('TOM (panel-specific scale)')
    ax.set_title(name)
    ax.grid(False)
fig.suptitle('TOM: Violin Plot + Individual Sample Values', fontsize=13, fontweight='bold')
fig.text(.5, 1.05, 'Violin = distribution shape; dots = observed samples', ha='center', fontsize=10)
violin_file = output_dir / 'Part2_TOM_Violin_Points.png'
fig.savefig(violin_file, dpi=300, bbox_inches='tight')
plt.show()
print('Saved:', violin_file)

**Remember:** Transformations do not guarantee normality;
1. Transformations can reduce right-skewness, but do not guarantee normality.
2. Compare skewness within each transformed scale; lower |skewness| means greater symmetry.
3. Boxplot outliers are observations for investigation, not automatic removal.
4. Violin width shows estimated density, whereas dots show individual measurements.
5. No TOM values are changed in the original Excel file.

# Part 3 — Spectral exploratory analysis

Understand the mean spectrum, the spread across samples, and how wavelength-by-wavelength reflectance relates to original TOM. These are **exploratory** correlations, not a supervised wavelength selection for model validation.

### 3.1 Read raw spectra

Every row is a sediment/soil sample; every selected wavelength is a predictor column. This part retains all source rows and handles non-finite measurements **at each wavelength**, without removing entire samples.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# Set this only if Part 3 is run without Parts 1–2.
project_dir = INPUT_DIR
p3_file = find_input_file('Pascal_Final.xlsx', pattern='Pascal_Final*.xlsx')
p3_output = STUDENT_OUTPUT_DIR / 'Part3_Outputs'
p3_output.mkdir(parents=True, exist_ok=True)
p3_wl = np.arange(360, 2501)  # Alternative: np.arange(360, 701)
p3_data = pd.read_excel(p3_file, sheet_name='Sheet1')
p3_data = normalize_spectral_columns(p3_data)
p3_data = p3_data.rename(columns={'TOM_%LOI': 'TOM'})
p3_cols = [str(x) for x in p3_wl]
p3_cols = [c if c in p3_data else 'X'+c for c in p3_cols]
p3_missing = [c for c in p3_cols+['TOM'] if c not in p3_data]
if p3_missing:
    raise ValueError(f'Missing required columns: {p3_missing[:10]}')
p3_R = p3_data[p3_cols].apply(pd.to_numeric, errors='coerce').to_numpy(dtype=float)
p3_R[~np.isfinite(p3_R)] = np.nan
p3_TOM = pd.to_numeric(p3_data['TOM'], errors='coerce').to_numpy(dtype=float)
p3_TOM[~np.isfinite(p3_TOM)] = np.nan
if p3_R.shape[0] < 3:
    raise ValueError('At least three samples are needed.')
print(f'Samples: {len(p3_R)} | Wavelengths: {len(p3_wl)}')
print(f'Missing reflectance cells: {np.isnan(p3_R).sum()} | Missing TOM: {np.isnan(p3_TOM).sum()}')

### 3.2 Calculate the statistics used by the graphs

In [ ]:
p3_n = np.sum(np.isfinite(p3_R), axis=0)
p3_mean = np.full(len(p3_wl), np.nan)
p3_sd = np.full(len(p3_wl), np.nan)
p3_min = np.full(len(p3_wl), np.nan)
p3_max = np.full(len(p3_wl), np.nan)
p3_corr = np.full(len(p3_wl), np.nan)
p3_pairs = np.zeros(len(p3_wl), dtype=int)
for j in range(len(p3_wl)):
    x = p3_R[:, j]
    valid = x[np.isfinite(x)]
    if valid.size:
        p3_mean[j], p3_min[j], p3_max[j] = valid.mean(), valid.min(), valid.max()
    if valid.size >= 2:
        p3_sd[j] = valid.std(ddof=1)
    keep = np.isfinite(x) & np.isfinite(p3_TOM)
    p3_pairs[j] = keep.sum()
    if keep.sum() >= 3 and x[keep].std(ddof=1) > 0 and p3_TOM[keep].std(ddof=1) > 0:
        p3_corr[j] = np.corrcoef(x[keep], p3_TOM[keep])[0, 1]

p3_results = pd.DataFrame({
    'Wavelength_nm': p3_wl, 'N_reflectance': p3_n,
    'Mean_reflectance': p3_mean, 'SD_reflectance': p3_sd,
    'Min_reflectance': p3_min, 'Max_reflectance': p3_max,
    'N_pairs_for_correlation': p3_pairs,
    'Pearson_r_with_TOM': p3_corr,
    'Abs_Pearson_r_with_TOM': np.abs(p3_corr),
})
display(p3_results.head())
# OPTIONAL output (commented out, as in the uploaded R script):
# p3_results.to_excel(p3_output / 'Part3_Spectral_EDA_Four_Graphs.xlsx', index=False)

### 3.3 Graph 1 — Mean spectrum ± one standard deviation

The line represents average reflectance; the blue region represents **mean ± sample SD**. This region describes the observed spread, **not a confidence interval**.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(p3_wl, p3_mean-p3_sd, p3_mean+p3_sd, color='skyblue', alpha=0.45)
ax.plot(p3_wl, p3_mean, color='#00008B', linewidth=1.3)
ax.set(title='Mean Raw Spectrum with +/- 1 SD Band', xlabel='Wavelength (nm)', ylabel='Reflectance')
ax.text(0.5, 1.07, 'Blue line = mean; shaded band = mean +/- sample SD', transform=ax.transAxes, ha='center', fontsize=10)
ax.grid(alpha=0.18)
fig.tight_layout()
fig.savefig(p3_output/'Part3_01_Mean_PlusMinus_SD.png', dpi=300)
plt.show()

**Discuss:** Where is the shaded SD band widest? Does large variability imply a wavelength predicts TOM well? (Not necessarily.)

### 3.4 Graph 2 — Minimum–maximum spectral envelope

The orange band covers the minimum and maximum measured reflectance at each wavelength. Extreme values may reflect real sample differences, measurement artifacts, or noise.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.fill_between(p3_wl, p3_min, p3_max, color='orange', alpha=0.25)
ax.plot(p3_wl, p3_min, color='#CC7000', linewidth=0.6)
ax.plot(p3_wl, p3_max, color='#CC7000', linewidth=0.6)
ax.plot(p3_wl, p3_mean, color='#00008B', linewidth=1.2)
ax.set(title='Raw Spectral Envelope: Minimum and Maximum', xlabel='Wavelength (nm)', ylabel='Reflectance')
ax.text(0.5, 1.07, 'Orange band = observed range; blue line = mean', transform=ax.transAxes, ha='center', fontsize=10)
ax.grid(alpha=0.18)
fig.tight_layout()
fig.savefig(p3_output/'Part3_02_Min_Max_Envelope.png', dpi=300)
plt.show()

### 3.5 Graph 3 — Signed Pearson correlation (r)

Positive and negative correlations show **direction** of linear association between reflectance at each wavelength and original TOM. Correlation does not establish causation.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.axhline(0, color='gray', linestyle='--', linewidth=0.9)
ax.plot(p3_wl, p3_corr, color='#CD3333', linewidth=1.15)
ax.set(ylim=(-1,1), title='Correlation Spectrum: Reflectance vs TOM', xlabel='Wavelength (nm)', ylabel='Pearson correlation (r)')
ax.text(0.5, 1.07, 'Pearson r at each wavelength using raw TOM (% LOI)', transform=ax.transAxes, ha='center', fontsize=10)
ax.grid(alpha=0.18)
fig.tight_layout()
fig.savefig(p3_output/'Part3_03_Pearson_Correlation_TOM.png', dpi=300)
plt.show()

### 3.6 Graph 4 — Absolute Pearson correlation (|r|)

Taking the absolute value ignores the sign: r = −0.8 and r = +0.8 both have |r| = 0.8. A large |r| suggests a strong **univariate linear association**, not necessarily high multivariate model importance.

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(p3_wl, np.abs(p3_corr), color='#1874CD', linewidth=1.15)
ax.set(ylim=(0,1), title='Absolute Correlation: |r| vs Wavelength', xlabel='Wavelength (nm)', ylabel='Absolute Pearson correlation (|r|)')
ax.text(0.5, 1.07, 'Higher |r| = stronger linear association with TOM, regardless of sign', transform=ax.transAxes, ha='center', fontsize=10)
ax.grid(alpha=0.18)
fig.tight_layout()
fig.savefig(p3_output/'Part3_04_Absolute_Pearson_Correlation_TOM.png', dpi=300)
plt.show()
print('Part 3: four PNG figures saved to', p3_output)

# Part 4 — Spectral quality control and outlier investigation

The original R script examines **three different QC signals**: high-frequency noise, sharp spectral changes, and PCA score distance. After the three independent screens, it combines their flags in an Excel report. **A flag means investigate, not delete.** The original input Excel file is not changed.

**Note:** Detector phase errors at 1000 ± 50 nm and 1800 ± 50 nm are excluded from the generic sharp-change screen; the separate detector-transition screen is disabled in the source R script.

### 4.1 Read raw spectra and set QC thresholds

A spectrum is eligible for numerical QC if at least 95% of its channels are finite. Missing channels are linearly interpolated **only in a QC working copy**. Figures always use the measured raw spectra.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import median_filter
from scipy.stats import chi2
from sklearn.decomposition import PCA
from IPython.display import display

project_dir = INPUT_DIR
p4_file = find_input_file('Pascal_Final.xlsx', pattern='Pascal_Final*.xlsx')
p4_output = STUDENT_OUTPUT_DIR / 'Part4_Spectral_QC'
p4_output.mkdir(parents=True, exist_ok=True)
p4_wl = np.arange(360, 2501)
P4_MIN_VALID_FRACTION = 0.95
P4_RUNNING_MEDIAN_K = 7
P4_NOISE_ROBUST_Z = 6
P4_DERIVATIVE_ROBUST_Z = 8
P4_JOIN_CENTERS = (1000,1800)
P4_JOIN_HALFWIDTH = 50
P4_PCA_MAX_COMPONENTS = 5
P4_PCA_CUMVAR_GOAL = 0.95
P4_PCA_ALPHA = 0.01

p4_data = pd.read_excel(p4_file, sheet_name='Sheet1')
p4_data = normalize_spectral_columns(p4_data)
p4_data = p4_data.rename(columns={'TOM_%LOI':'TOM'})
p4_cols = [str(w) if str(w) in p4_data else 'X'+str(w) for w in p4_wl]
p4_missing = [c for c in ['id',*p4_cols] if c not in p4_data]
if p4_missing: raise ValueError(f'Missing columns: {p4_missing[:10]}')
p4_ids = p4_data['id'].astype('string').fillna('').astype(str).to_numpy()
if any(not v.strip() for v in p4_ids): print('WARNING: Some sample IDs are missing.')
if len(set(p4_ids)) != len(p4_ids): print('WARNING: Duplicate sample IDs found.')
p4_raw = p4_data[p4_cols].apply(pd.to_numeric,errors='coerce').to_numpy(float)
p4_raw[~np.isfinite(p4_raw)] = np.nan
p4_n, p4_p = p4_raw.shape
if p4_n < 5: raise ValueError('At least five samples needed.')
p4_eligible = np.mean(np.isfinite(p4_raw),axis=1) >= P4_MIN_VALID_FRACTION
if p4_eligible.sum() < 5: raise ValueError('Fewer than five sufficiently complete spectra.')
p4_qc = p4_raw[p4_eligible].copy()
p4_qc_ids = p4_ids[p4_eligible]
p4_rows = np.flatnonzero(p4_eligible) + 1  # 1-based Excel data-row order
for i in range(len(p4_qc)):
    x=p4_qc[i]
    valid=np.isfinite(x)
    if not valid.all():
        p4_qc[i]=np.interp(p4_wl,p4_wl[valid],x[valid])
if not np.isfinite(p4_qc).all(): raise ValueError('Missing values remain after QC interpolation.')
p4_tom = pd.to_numeric(p4_data.get('TOM',pd.Series(np.nan,index=p4_data.index)),errors='coerce').to_numpy(float)
p4_tom[~np.isfinite(p4_tom)] = np.nan
print(f'Total samples: {p4_n} | QC assessed: {sum(p4_eligible)} | Not assessed: {sum(~p4_eligible)}')

### 4.2 QC helpers

The robust Z-score compares each sample with the median and **median absolute deviation** (MAD). When the MAD is zero, the robust Z-score is undefined; the script reports `NaN` rather than inventing a zero score.

In [ ]:
def p4_robust_z(values):
    x=np.asarray(values, dtype=float)
    med=np.nanmedian(x)
    mad=np.nanmedian(np.abs(x-med))
    if not np.isfinite(mad) or mad<=0: return np.full(x.shape,np.nan)
    return 0.67448975*(x-med)/mad

def p4_show_flags(title, flags):
    selected=p4_qc_ids[np.asarray(flags, dtype=bool)]
    print(f'{title}: {len(selected)} flagged')
    if selected.size: print('Sample IDs:', ', '.join(selected))

def p4_plot_flagged(flags,title,filename):
    fig,ax=plt.subplots(figsize=(11,7))
    unflagged = p4_raw[p4_eligible][~flags]
    flagged = p4_raw[p4_eligible][flags]
    for i,curve in enumerate(unflagged):
        ax.plot(p4_wl,curve,color='gray',linewidth=0.45,alpha=0.45,label='No flag' if i==0 else None)
    for i,curve in enumerate(flagged):
        ax.plot(p4_wl,curve,color='#CD0000',linewidth=0.9,alpha=0.9,label='Flag: investigate' if i==0 else None)
    ax.set(title=title,xlabel='Wavelength (nm)',ylabel='Reflectance')
    if len(unflagged) or len(flagged): ax.legend(loc='best')
    fig.tight_layout()
    fig.savefig(p4_output/filename,dpi=300)
    plt.show()

### 4.3 Section 1 — High-frequency noise

Fit a seven-channel **running median** to each spectrum. The sample noise score is the **median** absolute raw-minus-smoothed residual across wavelengths (not the mean). Flag samples whose robust Z-score is at least 6.

In [ ]:
# Match R's runmed(..., endrule='median'): edge values reuse the median
# of the first/last complete k-point window rather than shrinking the window.
from scipy.ndimage import median_filter

p4_smooth = median_filter(p4_qc, size=(1,P4_RUNNING_MEDIAN_K), mode='nearest')
p4_half = (P4_RUNNING_MEDIAN_K-1)//2
p4_smooth[:, :p4_half] = p4_smooth[:, p4_half][:, None]
p4_smooth[:, -p4_half:] = p4_smooth[:, -p4_half-1][:, None]
p4_noise_score = np.median(np.abs(p4_qc-p4_smooth),axis=1)
p4_noise_z = p4_robust_z(p4_noise_score)
p4_flag_noise = np.isfinite(p4_noise_z) & (p4_noise_z >= P4_NOISE_ROBUST_Z)
print('Noise score summary:')
print(pd.Series(p4_noise_score).describe().round(5))
if np.isnan(p4_noise_z).all():
    print('Noise robust Z unavailable: median absolute deviation of scores is zero.')
p4_show_flags('High-frequency noise',p4_flag_noise)
p4_plot_flagged(p4_flag_noise, 'High-Frequency Noise: Flagged Spectra', 'Part4_High_Frequency_Noise_Flags.png')

### 4.4 Section 2 — Sharp spectral changes

Calculate the first derivative, compare **absolute derivative magnitudes** across samples separately at each wavelength, and take each sample's maximum robust Z-score outside the detector joins. Flag Z ≥ 8. Report the wavelength of the maximum and every wavelength crossing the threshold.

In [ ]:
# np.gradient uses central differences internally and one-sided differences at edges.
p4_dR = np.gradient(p4_qc,p4_wl,axis=1,edge_order=1)
p4_allowed = (p4_wl>=360)&(p4_wl<=2490)
for center in P4_JOIN_CENTERS:
    p4_allowed &= (np.abs(p4_wl-center)>P4_JOIN_HALFWIDTH)
p4_sharp_z = np.column_stack([p4_robust_z(np.abs(p4_dR[:,j])) for j in range(p4_p)])
p4_sharp_z[~np.isfinite(p4_sharp_z)] = np.nan
p4_sharp_max = np.full(len(p4_qc),np.nan)
p4_sharp_peak = np.full(len(p4_qc),np.nan)
p4_sharp_wavelengths = [None]*len(p4_qc)
for i in range(len(p4_qc)):
    z = p4_sharp_z[i].copy()
    z[~p4_allowed] = np.nan
    good = np.flatnonzero(np.isfinite(z))
    if good.size:
        peak=good[np.argmax(z[good])]
        p4_sharp_max[i] = z[peak]
        p4_sharp_peak[i] = p4_wl[peak]
        above=np.flatnonzero(np.isfinite(z)&(z>=P4_DERIVATIVE_ROBUST_Z))
        if above.size: p4_sharp_wavelengths[i] = ', '.join(map(str,p4_wl[above]))
p4_flag_sharp = np.isfinite(p4_sharp_max)&(p4_sharp_max>=P4_DERIVATIVE_ROBUST_Z)
p4_sharp_table = pd.DataFrame({
    'Sample_ID':p4_qc_ids[p4_flag_sharp],
    'Wavelength_Max_Z_nm':p4_sharp_peak[p4_flag_sharp],
    'Max_Sharp_Robust_Z':np.round(p4_sharp_max[p4_flag_sharp],2),
    'Wavelengths_Z_ge_8_nm':np.array(p4_sharp_wavelengths,dtype=object)[p4_flag_sharp],
})
display(p4_sharp_table)
p4_show_flags('Sharp spectral changes',p4_flag_sharp)
p4_plot_flagged(p4_flag_sharp,'Sharp Spectral Changes: Flagged Spectra','Part4_Sharp_Change_Flags.png')

### 4.5 Section 3 — PCA score-distance screening

Standardize each wavelength across samples using **sample SD** (`ddof=1`), then select roughly 400 evenly spaced channels. PCA is fitted to that matrix. Keep enough PCs to explain 95% variance if possible, with **at most five PCs** (at least two). Distances use PC variances and an approximate chi-square 99% cutoff. A distant spectrum can still be scientifically valid.

In [ ]:
p4_sd_cols = np.std(p4_qc,axis=0,ddof=1)
p4_keep_cols = np.isfinite(p4_sd_cols)&(p4_sd_cols>1e-12)
if p4_keep_cols.sum()<2: raise ValueError('Too few variable wavelengths for PCA.')
p4_good = p4_qc[:,p4_keep_cols]
p4_standard = (p4_good - p4_good.mean(axis=0)) / p4_good.std(axis=0,ddof=1)
p4_step = max(1,p4_standard.shape[1]//400)
p4_pca_data = p4_standard[:,::p4_step]
# Full PCA spectrum is needed to calculate the same total variance as R's prcomp().
p4_pca = PCA(svd_solver='full')
p4_all_scores = p4_pca.fit_transform(p4_pca_data)
p4_eigen = p4_pca.explained_variance_
p4_pos = np.flatnonzero(p4_eigen>1e-12)
if p4_pos.size<2: raise ValueError('PCA needs at least two nonzero components.')
p4_cumvar = np.cumsum(p4_eigen)/p4_eigen.sum()
p4_goal = int(np.searchsorted(p4_cumvar,P4_PCA_CUMVAR_GOAL)+1)
p4_k = min(max(2,p4_goal),P4_PCA_MAX_COMPONENTS,len(p4_pos))
p4_scores = p4_all_scores[:,:p4_k]
p4_dist = np.sqrt(np.sum((p4_scores**2)/p4_eigen[:p4_k],axis=1))
p4_cutoff = np.sqrt(chi2.ppf(1-P4_PCA_ALPHA,df=p4_k))
p4_flag_pca = p4_dist > p4_cutoff
p4_percentile99 = np.quantile(p4_dist,0.99)
print(f'PCA components retained: {p4_k}')
print(f'Cumulative explained variance: {100*p4_cumvar[p4_k-1]:.1f}%')
print(f'Approximate 99% distance cutoff: {p4_cutoff:.3f}')
print(f'Observed 99th percentile: {p4_percentile99:.3f}')
p4_show_flags('PCA score distance',p4_flag_pca)

### 4.6 PCA graph 1 — PC1 versus PC2

Each point represents one sample. The horizontal and vertical axes are its scores on the first two principal components. Red points exceed the multicomponent PCA-distance cutoff.

In [ ]:
fig,ax=plt.subplots(figsize=(11,7))
for is_flag,label,color in [(False,'No PCA flag','gray'),(True,'PCA flag','#CD0000')]:
    subset=p4_flag_pca==is_flag
    if np.any(subset):ax.scatter(p4_scores[subset,0],p4_scores[subset,1],c=color,s=32,alpha=.85,label=label)
ax.set(title='PCA Scores: PC1 versus PC2', xlabel='PC1 score',ylabel='PC2 score')
ax.text(.5,1.01, f'PC1 = {100*p4_eigen[0]/p4_eigen.sum():.1f}%; PC2 = {100*p4_eigen[1]/p4_eigen.sum():.1f}%', transform=ax.transAxes,ha='center',fontsize=10)
ax.legend(); ax.grid(alpha=.15);fig.tight_layout()
fig.savefig(p4_output/'Part4_PCA_PC1_PC2_Flags.png',dpi=300)
plt.show()

### 4.7 PCA graph 2 — PCA score distance by original Excel row

The horizontal axis retains the source data row order (starting at 1). The dashed line is the approximate 99% screening threshold.

In [ ]:
fig,ax=plt.subplots(figsize=(11,7))
for is_flag,label,color in [(False,'No','gray'),(True,'Yes','#CD0000')]:
    sel=p4_flag_pca==is_flag
    if np.any(sel):ax.scatter(p4_rows[sel],p4_dist[sel],c=color,s=20,label=label)
ax.axhline(p4_cutoff,color='red',linestyle='--')
ax.set(title='PCA Score Distance by Original Excel Row',xlabel='Excel row order (data rows)',ylabel='Variance-standardized PCA distance')
ax.text(.5,1.01,'Dashed line: approximate 99% screening threshold',transform=ax.transAxes,ha='center',fontsize=10)
ax.legend(title='PCA flag');fig.tight_layout()
fig.savefig(p4_output/'Part4_PCA_Distance.png',dpi=300)
plt.show()

### 4.8 PCA graph 3 — Flagged spectra

Review the original raw spectral curves of samples identified by the PCA-distance screen.

In [ ]:
p4_plot_flagged(p4_flag_pca,'PCA Distance: Flagged Spectra','Part4_PCA_Flagged_Spectra.png')

### 4.9 Combine flags and export QC report

Bring the results back to the **original sample row order**, including samples not eligible for numerical QC. Keep the flags separate so students can see which detector identified which sample. Save the original four-sheet Excel report and an overview plot.

In [ ]:
p4_results = pd.DataFrame({
    'Excel_Row': np.arange(1,p4_n+1), 'Sample_ID':p4_ids,
    'TOM_percent_LOI':p4_tom, 'Flag_High_Noise':pd.Series([pd.NA]*p4_n,dtype='boolean'),
    'Flag_Sharp_Change':pd.Series([pd.NA]*p4_n,dtype='boolean'),
    'Flag_PCA_Distance':pd.Series([pd.NA]*p4_n,dtype='boolean'),
    'Noise_Robust_Z':np.full(p4_n,np.nan),
    'Max_Sharp_Robust_Z':np.full(p4_n,np.nan),
    'Wavelength_Max_Sharp_Z_nm':np.full(p4_n,np.nan),
    'Wavelengths_Sharp_Z_ge_8_nm':[None]*p4_n,
    'PCA_Distance':np.full(p4_n,np.nan),
    'PCA_PC1':np.full(p4_n,np.nan), 'PCA_PC2':np.full(p4_n,np.nan),
    'Number_of_Flags':pd.Series([pd.NA]*p4_n,dtype='Int64'),
    'Problems':['Not assessed: insufficient spectral values']*p4_n,
})
p4_ix=np.flatnonzero(p4_eligible)
p4_results.loc[p4_ix,'Flag_High_Noise']=p4_flag_noise
p4_results.loc[p4_ix,'Flag_Sharp_Change']=p4_flag_sharp
p4_results.loc[p4_ix,'Flag_PCA_Distance']=p4_flag_pca
p4_results.loc[p4_ix,'Noise_Robust_Z']=p4_noise_z
p4_results.loc[p4_ix,'Max_Sharp_Robust_Z']=p4_sharp_max
p4_results.loc[p4_ix,'Wavelength_Max_Sharp_Z_nm']=p4_sharp_peak
p4_results.loc[p4_ix,'Wavelengths_Sharp_Z_ge_8_nm']=p4_sharp_wavelengths
p4_results.loc[p4_ix,'PCA_Distance']=p4_dist
p4_results.loc[p4_ix,'PCA_PC1']=p4_scores[:,0]
p4_results.loc[p4_ix,'PCA_PC2']=p4_scores[:,1]
p4_flags=np.column_stack([p4_flag_noise,p4_flag_sharp,p4_flag_pca])
p4_labels=['High-frequency noise','Sharp derivative','PCA score distance']
p4_results.loc[p4_ix,'Number_of_Flags']=p4_flags.sum(axis=1)
p4_results.loc[p4_ix,'Problems']=['; '.join(n for n,f in zip(p4_labels,row) if f) or 'None' for row in p4_flags]
p4_flagged = p4_results[p4_results['Number_of_Flags'].isna() | (p4_results['Number_of_Flags']>0)].copy()
p4_flagged['_unassessed']=p4_flagged['Number_of_Flags'].isna()
p4_flagged=p4_flagged.sort_values(['_unassessed','Number_of_Flags'],ascending=[True,False]).drop(columns='_unassessed')
p4_counts = pd.DataFrame({'QC_Category':['Not assessed (<95% valid)',*p4_labels],
   'Number_of_Samples':[sum(~p4_eligible),*p4_flags.sum(axis=0)]})
p4_settings = pd.DataFrame({'Setting':[
    'Input file','Wavelength range (nm)','Minimum valid wavelength fraction','Detector-transition QC',
    'Noise method','Noise robust Z threshold','Running median width',
    'Sharp derivative robust Z threshold','PCA PCs for distance',
    'Variance explained by selected PCs (%)','PCA chi-square screening cutoff','PCA empirical 99th percentile'],
    'Value':[p4_file.name,f'{min(p4_wl)}-{max(p4_wl)}',P4_MIN_VALID_FRACTION,'Disabled',
        'Median absolute running-median residual',P4_NOISE_ROBUST_Z,P4_RUNNING_MEDIAN_K,
        P4_DERIVATIVE_ROBUST_Z,p4_k,round(100*p4_cumvar[p4_k-1],2),round(p4_cutoff,4),round(p4_percentile99,4)]})
p4_excel = p4_output/'Part4_Pascal_Spectral_QC_Flags.xlsx'
with pd.ExcelWriter(p4_excel,engine='openpyxl') as writer:
    p4_results.to_excel(writer,sheet_name='All_Samples',index=False)
    p4_flagged.to_excel(writer,sheet_name='Flagged_Or_Not_Assessed',index=False)
    p4_counts.to_excel(writer,sheet_name='QC_Counts',index=False)
    p4_settings.to_excel(writer,sheet_name='Settings',index=False)
p4_any=p4_flag_noise|p4_flag_sharp|p4_flag_pca
p4_plot_flagged(p4_any,'Raw Spectra: Any QC Flag in Red','Part4_Raw_Spectra_Flags.png')
print('Total samples:',p4_n,'| Assessed:',sum(p4_eligible),'| Flagged:',sum(p4_any))
display(p4_counts)
print('Report and figures saved to:',p4_output)
print('Remember: flagged means investigate, not automatically delete.')

# Part 5 — Spectral preprocessing: SG smoothing, SNV and MSC

Reproduces the three **independent** methods in your R script. All methods start with **RAW** spectra; SNV is not applied after SG, and MSC is not applied after SNV. Each method has a separate before/after graph. SG has one additional single-sample overlay.
**Note** In predictive modeling, an MSC reference must be fitted on training spectra only.

### 5.1 Read complete raw spectra and choose 12 examples

The preprocessing calculations use all complete spectra. The figures display 12 evenly spaced samples in their original source order to reduce visual clutter.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.signal import savgol_filter

project_dir = INPUT_DIR
p5_file = find_input_file('Pascal_Final.xlsx', pattern='Pascal_Final*.xlsx')
p5_output = STUDENT_OUTPUT_DIR / 'Part5_Preprocessing'
p5_output.mkdir(parents=True, exist_ok=True)
p5_wl=np.arange(360,2501)  # Change to np.arange(360,701) for VIS only
p5_data=pd.read_excel(p5_file,sheet_name='Sheet1')
p5_data.columns=p5_data.columns.map(str)
p5_cols=[str(w) if str(w) in p5_data else 'X'+str(w) for w in p5_wl]
p5_missing=[c for c in p5_cols if c not in p5_data]
if p5_missing: raise ValueError(f'Missing wavelength columns: {p5_missing[:10]}')
p5_all=p5_data[p5_cols].apply(pd.to_numeric,errors='coerce').to_numpy(float)
p5_ids=p5_data['id'].astype(str).to_numpy() if 'id' in p5_data else np.arange(1,len(p5_data)+1).astype(str)
p5_complete=np.all(np.isfinite(p5_all),axis=1)
p5_raw=p5_all[p5_complete].copy()
p5_ids=p5_ids[p5_complete]
print('Total samples:',len(p5_all))
print('Complete spectra used:',len(p5_raw))
print('Incomplete spectra excluded from figures:',sum(~p5_complete))
if len(p5_raw)<3: raise ValueError('Need at least three complete spectra')
p5_n_show=min(12,len(p5_raw))
# R round(seq(1,n,length.out=n_show)) is 1-based; np.rint uses ties-to-even.
p5_show=np.unique(np.rint(np.linspace(0,len(p5_raw)-1,p5_n_show)).astype(int))
print('Displayed sample IDs:',', '.join(p5_ids[p5_show]))

### 5.2 Reusable before-and-after graph

Use the same 12 examples on two panels. The y-axes are independent (`free_y` in the original R code), which matters for SNV.

In [ ]:
def p5_before_after(raw,processed,wl,method,filename,ylabel='Reflectance / processed value'):
    if raw.shape!=processed.shape or raw.shape[1]!=len(wl):
        raise ValueError('Raw and processed spectra must be aligned.')
    fig,axs=plt.subplots(2,1,figsize=(11,8),sharex=True)
    for ax,matrix,name in zip(axs,[raw,processed],['Before: raw',f'After: {method}']):
        for i in p5_show:
            ax.plot(wl,matrix[i],color='#1769AA',lw=.7,alpha=.65)
        ax.set_ylabel(ylabel)
        ax.set_title(name,fontsize=11,loc='left')
        ax.grid(alpha=.15)
    axs[-1].set_xlabel('Wavelength (nm)')
    fig.suptitle(f'Pascal spectra: Before and after {method}',fontweight='bold')
    fig.text(.5,.925,f'{len(p5_show)} representative spectra; original samples are not reordered',ha='center',fontsize=10)
    fig.tight_layout(rect=(0,0,1,.91))
    fig.savefig(p5_output/filename,dpi=300)
    plt.show()

### 5.3 Section 1 — Savitzky–Golay (SG) smoothing

At each wavelength, fit a local polynomial (order 2, 11-channel window). The original `prospectr::savitzkyGolay(..., m=0)` **removes five edge channels on each side**; Python's usual SciPy operation retains them, so we explicitly trim the edges to match R's output length.

In [ ]:
p5_window=11
p5_poly=2
# Use SciPy's local polynomial solution, then trim 5 channels at each side to
# reproduce the R prospectr::savitzkyGolay output alignment.
p5_sg_full=savgol_filter(p5_raw,window_length=p5_window,polyorder=p5_poly,deriv=0,axis=1,mode='interp')
p5_trim=(p5_window-1)//2
p5_sg=p5_sg_full[:,p5_trim:-p5_trim]
p5_sg_wl=p5_wl[p5_trim:-p5_trim]
p5_sg_raw=p5_raw[:,p5_trim:-p5_trim]
assert p5_sg.shape==p5_sg_raw.shape
p5_before_after(p5_sg_raw,p5_sg,p5_sg_wl,'SG smoothing (w=11, p=2)','Part5_01_SG_Before_After.png')

### 5.4 SG single-sample overlay

Displaying the raw and smoothed values on the **same axes** reveals subtle changes that can be hard to see when several curves overlap.

In [ ]:
p5_example=0
fig,ax=plt.subplots(figsize=(11,6))
ax.plot(p5_sg_wl,p5_sg_raw[p5_example],color='#4D4D4D',lw=1,label='Raw')
ax.plot(p5_sg_wl,p5_sg[p5_example],color='#E25B38',lw=1,label='SG smoothed')
ax.set(title=f'One spectrum: SG before vs after (ID {p5_ids[p5_example]})',xlabel='Wavelength (nm)',ylabel='Reflectance')
ax.legend(title='Signal'); ax.grid(alpha=.15);fig.tight_layout()
fig.savefig(p5_output/'Part5_01b_SG_One_Spectrum_Overlay.png',dpi=300)
plt.show()

### 5.5 Section 2 — Standard Normal Variate (SNV)

**For each sample separately**, subtract its mean across wavelengths and divide by its **sample SD** (`ddof=1`). SNV spectra can legitimately have negative values. Here SNV is applied to the original raw spectra, not SG output.

In [ ]:
p5_row_mean=p5_raw.mean(axis=1,keepdims=True)
p5_row_sd=p5_raw.std(axis=1,ddof=1,keepdims=True)
if np.any(~np.isfinite(p5_row_sd)|(p5_row_sd<=0)):
    raise ValueError('SNV cannot process a zero/invalid spectrum SD')
p5_snv=(p5_raw-p5_row_mean)/p5_row_sd
p5_before_after(p5_raw,p5_snv,p5_wl,'SNV','Part5_02_SNV_Before_After.png',
                ylabel='Raw reflectance / SNV-normalized value')
print('Example SNV sample mean:',round(p5_snv[0].mean(),6),
      '| Sample SD:',round(p5_snv[0].std(ddof=1),6))

### 5.6 Section 3 — Multiplicative Scatter Correction (MSC)

Calculate the **average raw spectrum** as reference. For each sample, regress its reflectance against that reference across wavelengths; remove the fitted intercept and slope. MSC is independently applied to raw spectra, not to SG- or SNV-processed spectra.

In [ ]:
p5_ref=p5_raw.mean(axis=0)
p5_ref_center=p5_ref-p5_ref.mean()
p5_ref_varsum=np.sum(p5_ref_center**2)
if not np.isfinite(p5_ref_varsum) or p5_ref_varsum<=0:
    raise ValueError('MSC reference has no variation')
p5_msc=np.empty_like(p5_raw)
for i,y in enumerate(p5_raw):
    slope=np.sum(p5_ref_center*(y-y.mean()))/p5_ref_varsum
    intercept=y.mean()-slope*p5_ref.mean()
    if not np.isfinite(slope) or abs(slope)<1e-12:
        raise ValueError(f'Invalid MSC slope for sample {p5_ids[i]}')
    p5_msc[i]=(y-intercept)/slope
p5_before_after(p5_raw,p5_msc,p5_wl,'MSC','Part5_03_MSC_Before_After.png',
                ylabel='Reflectance / MSC-corrected value')
print('Preprocessing complete. Figures saved to:',p5_output)
print('Remember: none of these preprocessing methods is guaranteed to improve TOM prediction.')

# Pascal soil spectroscopy — Part 6A — Reflectance SVR

SVR with fixed, externally provided sample IDs and fold assignments. No automatic deletion of samples or recreation of folds.

### Step 6A & 6B -1 — Mount Drive and import Python libraries
Run this once. All figures and Excel results are saved under the same Drive teaching folder.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from scipy.signal import savgol_filter
from sklearn.svm import SVR

# Part 6 uses the same Drive paths defined in the first setup cell.
PROJECT_DIR = INPUT_DIR
print('Instructor input folder:', PROJECT_DIR)
print('Student output folder:', STUDENT_OUTPUT_DIR)


### Step 6A & 6B -2 — Shared mathematical tools
We use original-scale metrics: Pearson r, correlation r², prediction R² (= 1 − SSE/SST), RMSE, MAE, and RPD. The model reproduces the default **internal standardization of X and y** used by R `e1071::svm`, after training-only min–max predictor scaling. This detail matters when comparing Python to R.

In [ ]:
# Shared functions for Parts 6A and 6B. Original data never modified.
def load_matching_data(input_filename, assignment_filename):
    data_path = find_input_file(input_filename)
    assn_path = find_input_file(assignment_filename)
    if not data_path.exists() or not assn_path.exists():
        raise FileNotFoundError(f'Missing {data_path if not data_path.exists() else assn_path}. Edit the filenames in the settings cell.')
    data = pd.read_excel(data_path, sheet_name='Sheet1')
    data = normalize_spectral_columns(data)
    data = data.rename(columns={'TOM_%LOI': 'TOM'})
    assignments = pd.read_excel(assn_path, sheet_name='Assignments')
    for col in ['id', 'TOM']:
        if col not in data: raise ValueError(f'Missing input column: {col}')
    if data['id'].isna().any() or data['id'].astype(str).duplicated().any():
        raise ValueError('Sample IDs must be present and unique.')
    data['TOM'] = pd.to_numeric(data['TOM'], errors='coerce')
    valid = np.isfinite(data['TOM']) & (data['TOM'] >= 0)
    print('Input rows:', len(data), '| Excluded for TOM:', int((~valid).sum()))
    data = data.loc[valid].copy().reset_index(drop=True)
    required = ['Sample_ID', 'Set', 'CV_Fold', 'Tuning_Fold']
    if any(x not in assignments for x in required):
        raise ValueError('Assignment workbook requires ' + ', '.join(required))
    def normal_id(series):
        return series.map(lambda z: str(int(z)) if pd.notna(z) and isinstance(z, (int, float, np.integer, np.floating)) and float(z).is_integer() else str(z).strip())
    ids = normal_id(data['id'])
    assignments['Sample_ID'] = normal_id(assignments['Sample_ID'])
    if ids.duplicated().any() or assignments['Sample_ID'].duplicated().any() or set(ids) != set(assignments.Sample_ID) or len(ids) != len(assignments):
        raise ValueError(f'ID MISMATCH: {len(ids)} eligible dataset IDs vs {len(assignments)} assignment IDs. Choose a matching dataset AND assignment workbook; do not regenerate or drop rows.')
    aligned = assignments.set_index('Sample_ID').loc[ids.to_list()].reset_index()
    if not aligned.Set.isin(['Train', 'Test']).all() or not {'Train','Test'}.issubset(set(aligned.Set)):
        raise ValueError('Train/Test assignments are missing or invalid.')
    tr = aligned.Set.eq('Train').to_numpy()
    for col, n in [('CV_Fold', 10), ('Tuning_Fold', 5)]:
        folds = pd.to_numeric(aligned.loc[tr, col], errors='coerce')
        if folds.isna().any() or set(folds.astype(int)) != set(range(1, n+1)) or not (folds == folds.astype(int)).all():
            raise ValueError(f'Invalid or empty training folds: {col}')
        if aligned.loc[~tr, col].notna().any():
            raise ValueError(f'Test samples have nonempty {col}.')
    return data, aligned, ids.to_numpy(), tr


def target_transform(y, mode):
    y = np.asarray(y, dtype=float)
    if mode == 'RAW': return y
    if mode == 'SQRT': return np.sqrt(y)
    if mode == 'LOG1P': return np.log1p(y)
    if mode == 'LOG':
        if np.any(y <= 0): raise ValueError('LOG requires positive TOM; do not silently drop zero observations.')
        return np.log(y)
    raise ValueError('Invalid target transform.')


def inverse_target(y, mode):
    if mode == 'RAW': return y
    if mode == 'SQRT': return y**2
    if mode == 'LOG1P': return np.expm1(y)
    if mode == 'LOG': return np.exp(y)
    raise ValueError(mode)


def original_metrics(obs, pred):
    a, b = np.asarray(obs, dtype=float), np.asarray(pred, dtype=float)
    if not np.all(np.isfinite(a)) or not np.all(np.isfinite(b)): raise ValueError('Nonfinite observations or predictions')
    rmse = np.sqrt(np.mean((a-b)**2)); sd = np.std(a, ddof=1) if len(a)>1 else np.nan
    r = np.corrcoef(a, b)[0,1] if len(a)>1 and sd>0 and np.std(b,ddof=1)>0 else np.nan
    sst = np.sum((a-a.mean())**2)
    return dict(Pearson_r=r, R2_Correlation=r**2, R2_Determination=1-np.sum((a-b)**2)/sst if sst>0 else np.nan,
                RMSE=rmse, MAE=np.mean(np.abs(a-b)), RPD=sd/rmse if rmse>0 else np.nan)

METRICS = ['Pearson_r','R2_Correlation','R2_Determination','RMSE','MAE','RPD']


def snv(x):
    sd = x.std(axis=1, ddof=1)
    if np.any(sd <= 0): raise ValueError('Zero sample spectrum SD in SNV')
    return (x - x.mean(axis=1,keepdims=True))/sd[:,None]


def msc(train, val):
    reference = train.mean(axis=0)
    refc = reference-reference.mean()
    denom = np.dot(refc,refc)
    if denom <= 0: raise ValueError('MSC reference has no variation')
    def correct(x):
        mu = x.mean(axis=1)
        slope = ((x-mu[:,None]) @ refc)/denom
        if np.any(np.abs(slope)<1e-12): raise ValueError('Invalid MSC slope')
        intercept = mu - slope*reference.mean()
        return (x-intercept[:,None])/slope[:,None]
    return correct(train), correct(val)


def preprocess(train, val, method, spectral=True):
    a, b = train.copy(), val.copy()
    if method == 'RAW': pass
    elif not spectral: raise ValueError('Color coefficients are not ordered wavelengths; use RAW only.')
    elif method == 'SNV': a,b=snv(a),snv(b)
    elif method == 'MSC': a,b=msc(a,b)
    elif method == 'd1': a,b=np.diff(a,axis=1),np.diff(b,axis=1)
    elif method in ['SG-d1-SNV','SG-d1-MSC','SG-SNV-MSC']:
        derivative = 0 if method == 'SG-SNV-MSC' else 1
        # scipy savgol_filter(interp) preserves endpoints; prospectr trims 5 columns.
        a = savgol_filter(a, 11, 2, deriv=derivative, axis=1, delta=1.0)[:,5:-5]
        b = savgol_filter(b, 11, 2, deriv=derivative, axis=1, delta=1.0)[:,5:-5]
        if method == 'SG-d1-SNV': a,b=snv(a),snv(b)
        elif method == 'SG-d1-MSC': a,b=msc(a,b)
        else: a,b=msc(snv(a),snv(b))
    else: raise ValueError(method)
    return a,b


def fit_svr(xtrain, ytrain, cost, epsilon):
    # e1071::svm defaults scale=TRUE: after R-script train-only minmax,
    # it standardizes BOTH the predictors and regression target on the fit subset.
    # Reproduce with training mean and sample SD (ddof=1).
    lo=xtrain.min(axis=0); rng=xtrain.max(axis=0)-lo
    span=np.where(np.isclose(rng,0,rtol=1e-8,atol=1e-8),1.0,rng)
    mm=(xtrain-lo)/span
    mm[:,np.isclose(rng,0,rtol=1e-8,atol=1e-8)]=0
    xmean=mm.mean(axis=0); xsd=mm.std(axis=0,ddof=1)
    xsd[xsd<=1e-12]=1.0
    ymean=ytrain.mean(); ysd=ytrain.std(ddof=1)
    if ysd<=0: raise ValueError('Constant training TOM target')
    model=SVR(kernel='linear',C=cost,epsilon=epsilon, tol=0.001)
    model.fit((mm-xmean)/xsd,(ytrain-ymean)/ysd)
    return (model,lo,span,rng,xmean,xsd,ymean,ysd)


def predict_svr(fit,x):
    model,lo,span,rng,xmean,xsd,ymean,ysd=fit
    mm=(x-lo)/span
    mm[:,np.isclose(rng,0,rtol=1e-8,atol=1e-8)]=0
    return model.predict((mm-xmean)/xsd)*ysd+ymean


def plot_pred(obs, pred, metrics, title, outpath):
    fig,ax=plt.subplots(figsize=(8.8,7.2))
    obs=np.asarray(obs);pred=np.asarray(pred)
    lo,hi=np.min(np.r_[obs,pred]),np.max(np.r_[obs,pred]); pad=max((hi-lo)*0.06,0.1)
    limits=[lo-pad,hi+pad]
    ax.scatter(obs,pred,color='#2474A6',alpha=.68,s=36)
    ax.plot(limits,limits,'--',color='#D05A45',linewidth=2)
    ax.set(xlim=limits,ylim=limits,xlabel='Observed TOM (% LOI)',ylabel='Predicted TOM (% LOI)',title=title)
    ax.set_aspect('equal', adjustable='box')
    text=f'N = {len(obs)}\nDetermination R² = {metrics["R2_Determination"]:.2f}\nRMSE = {metrics["RMSE"]:.2f}%\nMAE = {metrics["MAE"]:.2f}%\nRPD = {metrics["RPD"]:.2f}'
    ax.text(.03,.97,text.replace('\\n','\n'),transform=ax.transAxes,ha='left',va='top',fontsize=10)
    fig.tight_layout();fig.savefig(outpath,dpi=250);plt.show();plt.close(fig)
    print('Saved:',outpath)


def summarize_folds(df, col, prefix):
    out=df.copy();out.insert(0,'Set',[prefix]*len(df))
    means={m:df[m].mean() for m in METRICS};means[col]=np.nan;means['Set']=prefix+'_Mean'
    return pd.concat([out,pd.DataFrame([means])],ignore_index=True)


### Step 6A & 6B -3 — Modeling pipeline
Tuning: five fixed folds, mean original-scale RMSE. Validation: ten fixed folds. Independent test: predictions from all ten CV-trained models, averaged **after inverse TOM transformation**. Models are never fitted to test data.

In [ ]:
def run_scenario(X, y_original, ids, train_mask, assignments, target_mode, methods,
                 costs, epsilons, out_dir, filename_prefix, spectral=True):
    out_dir.mkdir(parents=True,exist_ok=True)
    y=target_transform(y_original,target_mode)
    ixtrain=np.flatnonzero(train_mask); ixtest=np.flatnonzero(~train_mask)
    xtr=X[ixtrain]; xte=X[ixtest]; ytr=y[ixtrain]; yte=y[ixtest]
    train_ids=ids[ixtrain]; test_ids=ids[ixtest]
    cv_folds=assignments.loc[train_mask,'CV_Fold'].to_numpy(dtype=int)
    tune_folds=assignments.loc[train_mask,'Tuning_Fold'].to_numpy(dtype=int)
    print('Training:',len(ixtrain),'| independent test:',len(ixtest))
    for pp in methods:
        print('\n=== Scenario:',filename_prefix,pp,target_mode,'===')
        tuning=[]
        for C in costs:
            for epsilon in epsilons:
                rmses=[]
                for fold in range(1,6):
                    a=tune_folds!=fold; b=~a
                    xx, vv=preprocess(xtr[a],xtr[b],pp,spectral)
                    model=fit_svr(xx,ytr[a],C,epsilon)
                    pred=inverse_target(predict_svr(model,vv),target_mode)
                    actual=inverse_target(ytr[b],target_mode)
                    rmses.append(original_metrics(actual,pred)['RMSE'])
                tuning.append(dict(cost=C,epsilon=epsilon,RMSE_CV_Original_TOM=np.mean(rmses)))
        tuning_df=pd.DataFrame(tuning)
        best=tuning_df.iloc[tuning_df.RMSE_CV_Original_TOM.idxmin()]
        C=float(best.cost);epsilon=float(best.epsilon)
        print(f'Best cost={C:g}, epsilon={epsilon:g}, mean 5-fold original TOM RMSE={best.RMSE_CV_Original_TOM:.3f}')
        val_metrics=[];tr_metrics=[];cv_rows=[];test_model_metrics=[];test_preds=[]
        for fold in range(1,11):
            is_val=cv_folds==fold; is_train=~is_val
            xfit,xval=preprocess(xtr[is_train],xtr[is_val],pp,spectral)
            model=fit_svr(xfit,ytr[is_train],C,epsilon)
            p_val=inverse_target(predict_svr(model,xval),target_mode)
            p_fit=inverse_target(predict_svr(model,xfit),target_mode)
            val_metrics.append(dict(Fold=fold,**original_metrics(inverse_target(ytr[is_val],target_mode),p_val)))
            tr_metrics.append(dict(Fold=fold,**original_metrics(inverse_target(ytr[is_train],target_mode),p_fit)))
            cv_rows.append(pd.DataFrame({'Sample_ID':train_ids[is_val],'Fold':fold,
                        'Actual_TOM':inverse_target(ytr[is_val],target_mode),'Predicted_TOM':p_val}))
            # The same saved fold model, trained without the validation fold, predicts the held-out independent test.
            _,xtest_processed=preprocess(xtr[is_train],xte,pp,spectral)
            p_test=inverse_target(predict_svr(model,xtest_processed),target_mode)
            test_preds.append(p_test)
            test_model_metrics.append(dict(Model=fold,**original_metrics(inverse_target(yte,target_mode),p_test)))
        cv_df=pd.concat(cv_rows,ignore_index=True); val_df=pd.DataFrame(val_metrics); tr_df=pd.DataFrame(tr_metrics)
        per_model_df=pd.DataFrame(test_model_metrics)
        ptest=np.column_stack(test_preds)
        test_df=pd.DataFrame({'Sample_ID':test_ids,'Actual_TOM':inverse_target(yte,target_mode)})
        for i in range(10):test_df[f'Predicted_TOM_Model_{i+1}']=ptest[:,i]
        test_df['Predicted_TOM_Ensemble']=ptest.mean(axis=1)
        pooled=original_metrics(cv_df.Actual_TOM,cv_df.Predicted_TOM)
        ensemble=original_metrics(test_df.Actual_TOM,test_df.Predicted_TOM_Ensemble)
        print('Mean CV-fold determination R²:',round(val_df.R2_Determination.mean(),3))
        print('Pooled held-out CV determination R²:',round(pooled['R2_Determination'],3))
        print('Independent test ensemble determination R²:',round(ensemble['R2_Determination'],3))
        base=f'{filename_prefix}_{pp}_SVM_{target_mode}'
        plot_pred(cv_df.Actual_TOM.to_numpy(),cv_df.Predicted_TOM.to_numpy(),pooled,
                  '10-fold CV: held-out observed vs predicted TOM',out_dir/f'{base}_CV_Observed_vs_Predicted.png')
        plot_pred(test_df.Actual_TOM.to_numpy(),test_df.Predicted_TOM_Ensemble.to_numpy(),ensemble,
                  'Independent test: 10-model ensemble observed vs predicted TOM',
                  out_dir/f'{base}_Ensemble_Test_Observed_vs_Predicted.png')
        settings=pd.DataFrame({'Parameter':['Input type','Target transformation','Preprocessing','Kernel','Best cost','Best epsilon','Training N','Test N','Error units','Assignment file'],
                               'Value':[filename_prefix,target_mode,pp,'linear',C,epsilon,len(ixtrain),len(ixtest),'original TOM percentage points',ASSIGNMENT_FILENAME]})
        sheets={'CV_PerFold_Validation':summarize_folds(val_df,'Fold','CV_PerFold_Validation'),
                'CV_Pooled_HeldOut':pd.DataFrame([dict(Set='CV_Pooled_HeldOut',**pooled)]),
                'Train_PerFold_Check':summarize_folds(tr_df,'Fold','Train_PerFold_Check'),
                'Test_PerModel_10Models':per_model_df.assign(Set='Test_PerModel_10Models'),
                'Test_Ensemble_10Models':pd.DataFrame([dict(Set='Test_Ensemble_10Models',**ensemble)]),
                'CV_Validation_Predictions':cv_df,
                'Independent_Test_Predictions':test_df,
                'Hyperparameter_Tuning':tuning_df,
                'Settings':settings}
        path=out_dir/f'{base}_OriginalTOM_Metrics_CV_Test.xlsx'
        with pd.ExcelWriter(path,engine='openpyxl') as writer:
            for name,frame in sheets.items():frame.to_excel(writer,sheet_name=name,index=False)
        print('Saved results:',path)
        display(sheets['CV_Pooled_HeldOut'].round(3))
        display(sheets['Test_Ensemble_10Models'].round(3))


### Step 6A-4 — Settings for spectral reflectance
Default example: VIS 360–700 nm, raw reflectance, log(TOM+1), 108-sample dataset. **Do not edit the assignment file itself.**

In [ ]:
# PART 6A SETTINGS
# These example filenames match the 108-sample training subset supplied here.
# If your Drive has different names, change BOTH input and assignment filenames.
REFLECTANCE_FILENAME = 'Pascal_Final.xlsx'
ASSIGNMENT_FILENAME = 'Pascal_Fixed_Sample_and_Fold_Assignments.xlsx'
WAVELENGTH_START, WAVELENGTH_END = 360, 700  # switch to 2500 for full range
TARGET_MODE = 'LOG1P'  # RAW, SQRT, LOG1P, LOG
PREPROCESSING_METHODS = ['RAW']
# Additional options: ['RAW','SNV','MSC','d1','SG-d1-SNV','SG-d1-MSC','SG-SNV-MSC']
OUTPUT_DIR = STUDENT_OUTPUT_DIR / 'Part6A_SVR_Reflectance'


### Step 6A-5 — Load reflectance and verify matching IDs
The original Excel and assignments must contain exactly the same eligible sample IDs. ID mismatches raise an error, rather than changing the holdout or folding.

In [ ]:
data, assignments, ids, train_mask = load_matching_data(REFLECTANCE_FILENAME, ASSIGNMENT_FILENAME)
wavelengths = np.arange(WAVELENGTH_START, WAVELENGTH_END+1)
cols = [str(w) for w in wavelengths]
missing=[c for c in cols if c not in data]
if missing: raise ValueError(f'Missing reflectance channels: {missing[:10]}')
X=data[cols].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=float)
if not np.isfinite(X).all(): raise ValueError('Missing/nonfinite reflectance; do not silently drop samples')
y_original = data['TOM'].to_numpy(dtype=float)
print('6A — reflectance predictors:',X.shape[1], '| same fixed ID assignments')


### Step 6A-6 — Tune, evaluate and graph
Review the pooled ten-fold validation plot and independent 10-model ensemble plot. Two PNGs and a nine-sheet Excel results workbook are saved.

In [ ]:
run_scenario(X,y_original,ids,train_mask,assignments,TARGET_MODE,PREPROCESSING_METHODS,
             costs=[0.1,1,10],epsilons=[0.01,0.1,0.5],out_dir=OUTPUT_DIR,
             filename_prefix=f'{WAVELENGTH_START}-{WAVELENGTH_END}',spectral=True)

**CV pooled R²** summarizes one held-out prediction for each training sample. **Average fold R²** averages 10 separate validation R² values, so they need not match. **Independent test R²** uses only the external 20% test subset. RMSE and MAE are TOM *percentage-point* errors, not relative percentage errors. Results may show small numerical differences from R due to the underlying LIBSVM implementations and tolerances.

### Step 6B-4 — Settings for the color-coefficient file
**This part reads the color Excel workbook, NOT the spectral reflectance workbook.**

In [ ]:
# EDITABLE PART 6B SETTINGS — NOT THE REFLECTANCE EXCEL FILE
COLOR_FILENAME = 'Pascal_Color_Coef_No land name.xlsx'
ASSIGNMENT_FILENAME = 'Pascal_Fixed_Sample_and_Fold_Assignments.xlsx'
TARGET_MODE = 'LOG1P'
COLOR = ['X','Y','Z','x','y','L','u','v','a','b','h','c']
RGB = ['R','G','B']
RGB_INDICES = ['HRGB','SRGB','Hue','Coro','Red','Sat','Int']
FEATURE_GROUPS = {'COLOR':COLOR,'RGB':RGB,'RGB_INDICES':RGB_INDICES,
                  'RGB_PLUS_INDICES':RGB+RGB_INDICES,
                  'COLOR_PLUS_RGB':COLOR+RGB,'ALL':COLOR+RGB+RGB_INDICES}
FEATURE_SETS=['RGB_PLUS_INDICES']  # e.g. ['COLOR','RGB','COLOR_PLUS_RGB']
OUTPUT_DIR = STUDENT_OUTPUT_DIR / 'Part6B_SVR_Color'


### Step 6B-5 — Load color workbook and verify matching IDs
The same externally fixed sample IDs and train/test and CV folds must match the source color workbook exactly.

In [ ]:
data, assignments, ids, train_mask = load_matching_data(COLOR_FILENAME, ASSIGNMENT_FILENAME)
y_original=data['TOM'].to_numpy(dtype=float)
print('6B — color source:',COLOR_FILENAME)
print('Available feature groups:',', '.join(FEATURE_GROUPS))


### Step 6B-6 — Tune and evaluate selected feature groups


In [ ]:
for feature_name in FEATURE_SETS:
    if feature_name not in FEATURE_GROUPS: raise ValueError(f'Unknown feature group: {feature_name}')
    columns=FEATURE_GROUPS[feature_name]
    missing=[c for c in columns if c not in data.columns]
    if missing: raise ValueError(f'Missing color features: {missing}')
    X_color=data[columns].apply(pd.to_numeric,errors='coerce').to_numpy(dtype=float)
    if not np.isfinite(X_color).all(): raise ValueError('Nonfinite color features; correct source, no silent row removal')
    print('Feature group:',feature_name,'| variables:',', '.join(columns))
    run_scenario(X_color,y_original,ids,train_mask,assignments,TARGET_MODE,['RAW'],
                 costs=[0.01,0.05,0.1,0.5,1],epsilons=[0.01,0.05,0.1,0.5],
                 out_dir=OUTPUT_DIR,filename_prefix='Color_'+feature_name,spectral=False)

**CV pooled R²** summarizes one held-out prediction for each training sample. **Average fold R²** averages 10 separate validation R² values, so they need not match. **Independent test R²** uses only the external 20% test subset. RMSE and MAE are TOM *percentage-point* errors, not relative percentage errors. Results may show small numerical differences from R due to the underlying LIBSVM implementations and tolerances.

---
## Part 7 — Optional satellite connection: NASA EMIT imaging spectroscopy

Parts 1–6 use **laboratory/proximal VNIR–SWIR soil reflectance**, not satellite pixels. A satellite instrument additionally measures mixed pixels under varying atmospheric, illumination, surface-moisture, and observation conditions. Directly applying a lab-trained TOM model to EMIT pixels **without sensor-response harmonization, suitable calibration data, spatial support matching and external validation is not scientifically justified**.

[NASA EMIT Level 2A estimated surface reflectance](https://developers.google.com/earth-engine/datasets/catalog/NASA_EMIT_L2A_RFL) is available on Google Earth Engine as `NASA/EMIT/L2A/RFL` (approximately 60 m). It is a spaceborne **imaging spectrometer** operated from the International Space Station. This optional demonstration retrieves one observed pixel spectrum where an EMIT scene exists; it does **not** produce a soil property map or make a lab-to-space calibration claim.

**To run:** Enable the switch below, provide your **own Earth Engine Cloud project ID**, and choose a location/date with coverage. This section uses Earth Engine only; the soil workbook stays in Google Drive. Coverage, flags and quotas can prevent a valid observation.

In [ ]:
RUN_OPTIONAL_EMIT = False  # Change to True only after preparing your Earth Engine project
EE_PROJECT_ID = ''  # Your own GEE-enabled Google Cloud project ID, e.g. 'ee-yourusername'
EMIT_LON, EMIT_LAT = -121.4612, 38.3340  # Example location used by the EMIT catalog
EMIT_START, EMIT_END = '2022-08-09', '2026-09-01'

if RUN_OPTIONAL_EMIT:
    if not EE_PROJECT_ID.strip():
        raise ValueError('Enter your own EE_PROJECT_ID before requesting EMIT data.')
    import ee
    import matplotlib.pyplot as plt
    ee.Authenticate()
    ee.Initialize(project=EE_PROJECT_ID)
    point = ee.Geometry.Point([EMIT_LON, EMIT_LAT])
    scenes = (ee.ImageCollection('NASA/EMIT/L2A/RFL')
              .filterBounds(point).filterDate(EMIT_START, EMIT_END))
    n_scenes = scenes.size().getInfo()
    print('EMIT scenes intersecting your point:', n_scenes)
    if n_scenes == 0:
        print('No EMIT scene for the selected point/date. Choose another location or date.')
    else:
        scene = ee.Image(scenes.first())
        flags = scene.select(['aggregate_flag']).reduceRegion(
            reducer=ee.Reducer.first(), geometry=point, scale=60, maxPixels=1e6).getInfo()
        names = [b for b in scene.bandNames().getInfo() if b.startswith('reflectance_')]
        centers = scene.get('reflectance_wavelengths').getInfo()
        observed = scene.select(names).reduceRegion(
            reducer=ee.Reducer.first(), geometry=point, scale=60, maxPixels=1e6).getInfo()
        pairs = [(float(w), float(observed[b])) for w, b in zip(centers, names)
                 if observed.get(b) is not None and np.isfinite(observed[b])]
        print('Aggregate quality flag at this pixel:', flags.get('aggregate_flag'))
        print('Retrieved valid spectral channels:', len(pairs))
        if pairs:
            w, r = np.asarray(pairs).T
            plt.figure(figsize=(10, 4.8))
            plt.plot(w, r, color='navy', lw=1.1)
            plt.xlabel('Wavelength (nm)'); plt.ylabel('Estimated surface reflectance')
            plt.title('NASA EMIT observed pixel spectrum (interpret flags before use)')
            plt.grid(alpha=0.3); plt.tight_layout()
            plt.savefig(STUDENT_OUTPUT_DIR / 'Part7_EMIT_Pixel_Spectrum.png', dpi=220)
            plt.show()
        else:
            print('All spectral measurements masked/unavailable. Try another location/scene.')
else:
    print('Optional EMIT activity skipped. Set RUN_OPTIONAL_EMIT=True to enable.')


### Interpretation questions (discuss; no automatic satellite model transfer)

1. Identify wavelength ranges with distinctive absorption or reflectance features in the laboratory data. Which could be affected by atmospheric absorption in orbital observations?
2. Explain the differences among **SG smoothing**, **SNV** and **MSC**. Why must feature transformations derived from training data be fitted without validation/test leakage?
3. Differentiate Pearson correlation *r²* from predictive **R² = 1 − SSE/SST**. Why should the independent test result matter more than a cross-validation result used during model selection?
4. Explain why a 60 m EMIT pixel cannot be assumed equivalent to a point soil spectrum. Discuss vegetation cover, moisture, calibration transfer, sensor response, mixing and geolocation.

**Modeling caveat:** In the original Part 6 protocol, a shared five-fold tuning procedure selects SVR hyperparameters on the training set and a separate fixed ten-fold scheme reports predictions. This is **not fully nested cross-validation**; the reported ten-fold CV score may be optimistic as a selection estimate. The externally held-out test set is a stronger check, provided it truly remained unused during any modeling decisions. Do not treat prediction as causal evidence.

**For submission:** Review the class instructions on UM Learn. At minimum, retain your Colab copy and inspect the figures/reports saved under your personal Drive output folder. Instructor-shared source data are not submitted or redistributed.